In [ ]:
# Cell 1: Imports and Setup
"""
Minimal RAG System - Works with basic LangChain installation only
No external vector database dependencies required.
"""

import os
import os
from os import path, listdir
from dotenv import load_dotenv
from typing import List, Optional
import PyPDF2 
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import OpenAIEmbeddings

# New imports for Gradio and OpenAI
import gradio as gr
import openai
from openai import OpenAI

In [ ]:
# Cell 2: Load environment variables

load_dotenv(override=True)
openai = OpenAI()
api_key = os.getenv("OPENAI_API_KEY")

# Check the key

if not api_key:
    print("OPENAI_API_KEY not found in environment variables.")
else:
    print ("API Key found.")

In [ ]:
# Cell 3: Simple Vector Store
class SimpleVectorStore:
    def __init__(self, documents, embeddings):
        self.documents = documents
        self.doc_embeddings = [embeddings.embed_query(doc.page_content) for doc in documents]
    
    def similarity_search(self, query, embeddings, k=3):
        query_embedding = embeddings.embed_query(query)
        similarities = []
        
        for i, doc_embedding in enumerate(self.doc_embeddings):
            similarity = sum(a * b for a, b in zip(query_embedding, doc_embedding))
            similarities.append((similarity, i))
        
        similarities.sort(reverse=True)
        return [self.documents[i] for _, i in similarities[:k]]

In [ ]:
# Cell 4: Document Loading (MODIFIED VERSION)
def load_pdfs(resume_directory_path: Optional[str] = None) -> List[Document]:
    if resume_directory_path is None:
        resume_directory_path = "/Users/lkeira108/Downloads/Resume"
    
    if not path.exists(resume_directory_path):
        return []
    
    documents = []
    for filename in listdir(resume_directory_path):
        # Look for resume AND LinkedIn PDF
        if filename.endswith(".pdf") and ("KL" in filename.upper() or "LINKEDIN" in filename.upper()):
            file_path = path.join(resume_directory_path, filename)
            
            try:
                with open(file_path, 'rb') as pdf_file:
                    pdf_reader = PyPDF2.PdfReader(pdf_file)
                    
                    for page_num, page in enumerate(pdf_reader.pages):
                        page_text = page.extract_text()
                        if page_text.strip():
                            # Identify document type
                            doc_type = "linkedin" if "LINKEDIN" in filename.upper() else "resume"
                            
                            metadata = {
                                "filename": filename,
                                "page_number": page_num + 1,
                                "doc_type": doc_type  # Add document type
                            }
                            documents.append(Document(page_content=page_text, metadata=metadata))
            except Exception as e:
                print(f"Error: {filename}: {e}")
    
    return documents

In [ ]:
# Cell 5: RAG Chat System
class RAGChatSystem:
    def __init__(self):
        self.client = OpenAI()
        self.vector_store = None
        self.embeddings = OpenAIEmbeddings()
        
    def initialize_rag(self):
        # Load summary text
        try:
            with open("me/summary.txt", "r", encoding="utf-8") as f:
                self.summary = f.read()
        except FileNotFoundError:
            self.summary = ""
        
        # Load PDF documents
        documents = load_pdfs()
        if not documents:
            return "No documents found"
                
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        chunks = text_splitter.split_documents(documents)
        self.vector_store = SimpleVectorStore(chunks, self.embeddings)
        return "RAG system ready!"
        
    def query(self, question):
        if not self.vector_store:
            return "No context available"
        docs = self.vector_store.similarity_search(question, self.embeddings, k=3)
        return "\n".join([doc.page_content for doc in docs])[:800]
    
    def chat(self, message, history):
        """Chat function that combines RAG context with OpenAI."""
        try:
            # Get relevant context from RAG
            rag_context = self.query(message)
            
            # Combine summary and RAG context
            context = f"Summary: {self.summary}\n\nDetailed context: {rag_context}"
            name = "Keira"
                
            # System prompt
            system_prompt = f"""You ARE {name}. Use "I", "my", "me" - never third person.
You're answering questions about YOUR career and background on YOUR website.

Your background: {context}

Be professional, optimistic, and engaging, as if talking to a potential client or future employer who came across the website. If you don't know something, say "I'd be happy to provide details via email or phone number."
"""
            
            # Build messages
            messages = [{"role": "system", "content": system_prompt}]
            
            # Add recent history (last 10 messages to manage tokens)
            for msg in history[-10:]:
                if isinstance(msg, dict) and 'role' in msg:
                    messages.append(msg)
            
            messages.append({"role": "user", "content": message})
            
            # Call OpenAI
            response = self.client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=messages,
                max_tokens=400,
                temperature=0.7
            )
            
            bot_response = response.choices[0].message.content.strip()
            
            # Update history
            history.append({"role": "user", "content": message})
            history.append({"role": "assistant", "content": bot_response})
            
            return bot_response, history
            
        except Exception as e:
            error_msg = "I'm having technical difficulties. Please contact me via email for more information."
            history.append({"role": "user", "content": message})
            history.append({"role": "assistant", "content": error_msg})
            return error_msg, history

In [ ]:
# Cell 6: Gradio Interface
def create_gradio_interface():
    """Create and launch Gradio chat interface."""
    
    # Initialize the RAG chat system
    rag_chat = RAGChatSystem()
    
    # Initialize RAG system on startup
    init_message = rag_chat.initialize_rag()
    print(init_message)
    
# Define the chat function for Gradio
    def chat_fn(message, history):
        # CHANGE 1: Convert Gradio history format [(user, bot), ...] to OpenAI format [{"role": "user", "content": "..."}, ...]
        openai_history = []
        for user_msg, bot_msg in history:
            openai_history.append({"role": "user", "content": user_msg})
            openai_history.append({"role": "assistant", "content": bot_msg})
        
        # CHANGE 2: Call your chat function correctly and unpack both return values (response, updated_history)
        response, updated_openai_history = rag_chat.chat(message, openai_history)
        
        # CHANGE 3: Add the new exchange to history and return properly
        history.append((message, response))
        return history, ""
    
    # Create Gradio interface
    with gr.Blocks(title="Resume RAG Chat") as demo:
        gr.Markdown("# 💬 Keira's Resume RAG Chat Assistant")
        gr.Markdown("Ask questions about the candidate's resume!")
        
        chatbot = gr.Chatbot(
            value=[],
            elem_id="chatbot",
            bubble_full_width=False,
            height=500
        )
        
        with gr.Row():
            msg = gr.Textbox(
                container=False,
                scale=7
            )
            submit = gr.Button("Send", scale=1, variant="primary")
            clear = gr.Button("Clear", scale=1)
        
        # Event handlers
        msg.submit(chat_fn, [msg, chatbot], [chatbot, msg])
        submit.click(chat_fn, [msg, chatbot], [chatbot, msg])
        clear.click(lambda: ([], ""), None, [chatbot, msg], queue=False)
    
    return demo


In [ ]:
# Cell 7: Launch Interface
def launch_chat():
    """Launch the Gradio chat interface."""
    demo = create_gradio_interface()
    demo.launch(
        share=True             # Create public link
    )

In [ ]:
# Cell 8: Execute
if __name__ == "__main__":
    print("Starting Resume RAG Chat Interface...")
    print("Make sure you have set your OPENAI_API_KEY environment variable!")
    launch_chat()
else:
    print("Ready to launch!")
    print("Run: launch_chat() to start the interface")
    print("Or run individual components for testing")